# AIC 2026 B01 OCR/ASR partition worker

Open up to four copies and set `PHYSICAL_WORKER_ID` from 0 to `PHYSICAL_WORKER_COUNT - 1`. Physical executors split the two immutable logical worker slots without changing the partition-plan fingerprint. This notebook only runs its disjoint partition lane; it never finalizes artifacts.

In [ ]:
STAGE = 'ocr'  # change to 'asr' only after OCR coordination completes
PHYSICAL_WORKER_ID = 0  # use 0, 1, 2, 3 across four notebook copies
PHYSICAL_WORKER_COUNT = 4
WORKER_COUNT = 2  # immutable logical count stored in the partition plan
assert 0 <= PHYSICAL_WORKER_ID < PHYSICAL_WORKER_COUNT
WORKER_SLOT = PHYSICAL_WORKER_ID % WORKER_COUNT
EXECUTOR_SLOT = PHYSICAL_WORKER_ID // WORKER_COUNT
EXECUTOR_COUNT = ((PHYSICAL_WORKER_COUNT - 1 - WORKER_SLOT) // WORKER_COUNT) + 1
BATCH_ID = 'aic25-b1-reference'
EXPECTED_VIDEOS = 873
EXPECTED_CANONICAL_FRAMES = 176_707


In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/AIC2026/B01')
RAW_ROOT = DRIVE_ROOT / 'raw'
OUTPUT_ROOT = DRIVE_ROOT / 'out'
SOURCE_BUNDLE = DRIVE_ROOT / 'source' / 'aic2026-colab-source.zip'
SOURCE_SIDECAR = SOURCE_BUNDLE.with_suffix('.zip.sha256')
REPO_ROOT = Path('/content/aic2026')
SCRATCH_ROOT = Path('/content/aic2026-scratch')
PLAN_PATH = OUTPUT_ROOT / 'plans' / f'{STAGE}-partitions.json'

expected_source_sha = SOURCE_SIDECAR.read_text(encoding='ascii').split()[0]
source_digest = hashlib.sha256()
with SOURCE_BUNDLE.open('rb') as stream:
    for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
        source_digest.update(chunk)
assert source_digest.hexdigest() == expected_source_sha, 'Source fingerprint mismatch'
REPO_ROOT.mkdir(parents=True, exist_ok=True)
repo_root = REPO_ROOT.resolve()
with zipfile.ZipFile(SOURCE_BUNDLE) as archive:
    for member in archive.infolist():
        target = (REPO_ROOT / member.filename).resolve()
        if target != repo_root and repo_root not in target.parents:
            raise RuntimeError(f'Unsafe source member: {member.filename}')
        if member.is_dir():
            target.mkdir(parents=True, exist_ok=True)
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(member) as source, target.open('wb') as output:
            shutil.copyfileobj(source, output, length=1024 * 1024)
os.chdir(REPO_ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv==0.11.16'], check=True)
subprocess.run(['uv', 'python', 'install', '3.11'], check=True)
subprocess.run(['uv', 'sync', '--python', '3.11', '--group', 'competition', '--frozen'], check=True)
schema = subprocess.check_output(['uv', 'run', '--frozen', 'python', '-c', 'from src.ingest.cloud_artifacts import CLOUD_ARTIFACT_SCHEMA_VERSION; print(CLOUD_ARTIFACT_SCHEMA_VERSION)'], text=True).strip()
assert schema == '3', f'Progressive source bundle required, found schema {schema}'
subprocess.run(['uv', 'run', '--frozen', 'python', 'scripts/prepare_models.py', '--only', 'reference-ocr' if STAGE == 'ocr' else 'reference-asr'], check=True)
assert PLAN_PATH.is_file(), f'Missing immutable partition plan: {PLAN_PATH}'


In [ ]:
plan = json.loads(PLAN_PATH.read_text(encoding='utf-8'))
assert plan['contract'] == 'cloud-partition-plan-v1'
assert plan['batch_id'] == BATCH_ID and plan['stage'] == STAGE
assert plan['worker_count'] == WORKER_COUNT
assert plan['expected_videos'] == EXPECTED_VIDEOS
assert plan['expected_frames'] == EXPECTED_CANONICAL_FRAMES
logical_assigned = [part for part in plan['partitions'] if part['worker_slot'] == WORKER_SLOT]
assigned = logical_assigned[EXECUTOR_SLOT::EXECUTOR_COUNT]
assert assigned, f'No partitions assigned to physical worker {PHYSICAL_WORKER_ID}'
first_keyframe = sorted(RAW_ROOT.glob('Keyframes_L*.zip'))[0]
for part in assigned:
    command = [
        'uv', 'run', '--frozen', 'python', 'scripts/colab_provision_zip.py',
        '--maps-zip', str(RAW_ROOT / 'map-keyframes-aic25-b1.zip'),
        '--output-dir', str(OUTPUT_ROOT), '--batch-id', BATCH_ID,
        '--stages', STAGE, '--expected-videos', str(EXPECTED_VIDEOS),
        '--expected-frames', str(EXPECTED_CANONICAL_FRAMES),
        '--partition-manifest', str(PLAN_PATH),
        '--partition-id', part['partition_id'],
        '--worker-slot', str(WORKER_SLOT), '--worker-count', str(WORKER_COUNT),
        '--resume', '--device', 'cpu',
    ]
    if STAGE == 'ocr':
        command += ['--keyframes-zip', str(RAW_ROOT / part['archive'])]
    else:
        command += [
            '--keyframes-zip', str(first_keyframe),
            '--videos-zip', str(RAW_ROOT / part['archive']),
            '--ffmpeg-path', 'ffmpeg', '--scratch-dir', str(SCRATCH_ROOT),
        ]
    print(json.dumps({'starting': part['partition_id'], 'logical_slot': WORKER_SLOT, 'physical_worker_id': PHYSICAL_WORKER_ID}, sort_keys=True), flush=True)
    subprocess.run(command, check=True)
print(json.dumps({'state': 'worker_complete', 'stage': STAGE, 'logical_slot': WORKER_SLOT, 'physical_worker_id': PHYSICAL_WORKER_ID, 'partitions': len(assigned)}, sort_keys=True))
